In [ ]:
!pip install pingouin factor_analyzer -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [ ]:
import re
import pandas as pd
import numpy as np
import pingouin as pg

from scipy import stats
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from google.colab import files

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [ ]:
up = files.upload()

nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Blank Quiz (Responses 3 TIC).xlsx to Blank Quiz (Responses 3 TIC).xlsx
File terbaca: Blank Quiz (Responses 3 TIC).xlsx


In [ ]:
JUMLAH_KOLOM_DEMOGRAFI = 5

df = pd.read_excel(nama_file)
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))

Jumlah responden : 40
Jumlah butir     : 30
Data kosong      : 0


In [ ]:
skala = {
    "Sangat Tidak Setuju (STJ)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5
}

num = butir.apply(
    lambda kolom: kolom.astype("string").str.strip().map(skala)
).apply(pd.to_numeric, errors="coerce")

print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))

num.head()

Sel yang gagal dikonversi: 40


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Respon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Respon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Respon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Respon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Respon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,5,5,4,4,4,4,4,5,4,4,...,2,5,4,5,4,5,5,4,4,5
1,4,4,4,4,4,4,4,5,4,4,...,4,2,4,4,4,5,4,5,5,5
2,5,5,4,4,4,4,4,4,4,4,...,2,4,5,4,2,5,5,5,4,5
3,5,4,5,2,4,5,5,4,4,2,...,2,4,4,4,4,5,4,4,4,4
4,5,5,4,4,5,5,5,4,2,4,...,2,2,2,2,4,5,5,4,5,5


In [ ]:
dimensi = {}

for kolom in butir.columns:
    hasil = re.match(r"^(.*?)\s*\[", str(kolom))

    if hasil:
        nama = hasil.group(1).strip().rstrip(". ")
        dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik: 4 butir
Respon emosional: 5 butir


In [ ]:
n = len(num)

t = stats.t.ppf(0.975, n - 2)

r_tabel = t / np.sqrt((n - 2) + t**2)

print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

n = 40 | r tabel (α=0,05; df=38) = 0.312


In [ ]:
hasil_validitas = []

for idx, (nama, daftar) in enumerate(dimensi.items()):
    x = num[daftar].dropna()
    total = x.sum(axis=1)

    for i, kolom in enumerate(daftar, 1):
        data_uji = pd.concat(
            [x[kolom], total - x[kolom]], axis=1
        ).dropna()

        n = len(data_uji)

        if n < 3 or data_uji.iloc[:, 0].nunique() < 2 or data_uji.iloc[:, 1].nunique() < 2:
            r = np.nan
            p = np.nan
            r_koreksi = np.nan
            keterangan = "Tidak dapat dihitung"

        else:
            r, p = stats.pearsonr(
                x.loc[data_uji.index, kolom],
                total.loc[data_uji.index]
            )

            r_koreksi, _ = stats.pearsonr(
                data_uji.iloc[:, 0],
                data_uji.iloc[:, 1]
            )

            t = stats.t.ppf(0.975, n - 2)
            r_tabel = t / np.sqrt((n - 2) + t**2)

            keterangan = "Valid" if r > r_tabel else "Tidak valid"

        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"{chr(65 + idx)}{i}",
            "Jumlah data": n,
            "r hitung": round(r, 3) if pd.notna(r) else np.nan,
            "Corrected r": round(r_koreksi, 3) if pd.notna(r_koreksi) else np.nan,
            "p-value": round(p, 4) if pd.notna(p) else np.nan,
            "Keterangan": keterangan
        })

tabel_validitas = pd.DataFrame(hasil_validitas)

display(tabel_validitas)

,Dimensi,Butir,Jumlah data,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,A1,40,0.419,0.222,0.0071,Valid
1,Penggunaan AI pada tugas atau projek kuliah,A2,40,0.390,0.224,0.0129,Valid
2,Penggunaan AI pada tugas atau projek kuliah,A3,40,0.626,0.418,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,A4,40,0.506,0.205,0.0009,Valid
4,Penggunaan AI pada tugas atau projek kuliah,A5,40,0.793,0.597,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,A6,40,0.634,0.424,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,A7,40,0.231,-0.061,0.1510,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,40,0.578,0.437,0.0001,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,40,0.708,0.541,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,40,0.804,0.651,0.0000,Valid


In [ ]:
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (
        1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1)
    )


def kategori(a):
    return (
        "Sangat baik" if a >= 0.90 else
        "Baik" if a >= 0.80 else
        "Reliabel" if a >= 0.70 else
        "Tidak reliabel"
    )


hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)

    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a)
    })

    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(
                    cronbach_alpha(x.drop(columns=kolom)), 3
                )
            })


tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)

display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.550,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.678,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.808,Baik
3,Prokrastinasi akademik,4,0.865,Baik
4,Respon emosional,5,0.555,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,A1,0.531
1,Penggunaan AI pada tugas atau projek kuliah,A2,0.533
2,Penggunaan AI pada tugas atau projek kuliah,A3,0.456
3,Penggunaan AI pada tugas atau projek kuliah,A4,0.547
4,Penggunaan AI pada tugas atau projek kuliah,A5,0.341
5,Penggunaan AI pada tugas atau projek kuliah,A6,0.453
6,Penggunaan AI pada tugas atau projek kuliah,A7,0.639
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.637
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.597
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.550


In [ ]:
gabung = tabel_validitas.merge(
    tabel_alpha_dihapus,
    on=["Dimensi", "Butir"],
    how="left"
)

display(gabung)

,Dimensi,Butir,Jumlah data,r hitung,Corrected r,p-value,Keterangan,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,A1,40,0.419,0.222,0.0071,Valid,0.531
1,Penggunaan AI pada tugas atau projek kuliah,A2,40,0.390,0.224,0.0129,Valid,0.533
2,Penggunaan AI pada tugas atau projek kuliah,A3,40,0.626,0.418,0.0000,Valid,0.456
3,Penggunaan AI pada tugas atau projek kuliah,A4,40,0.506,0.205,0.0009,Valid,0.547
4,Penggunaan AI pada tugas atau projek kuliah,A5,40,0.793,0.597,0.0000,Valid,0.341
5,Penggunaan AI pada tugas atau projek kuliah,A6,40,0.634,0.424,0.0000,Valid,0.453
6,Penggunaan AI pada tugas atau projek kuliah,A7,40,0.231,-0.061,0.1510,Tidak valid,0.639
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,40,0.578,0.437,0.0001,Valid,0.637
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,40,0.708,0.541,0.0000,Valid,0.597
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,40,0.804,0.651,0.0000,Valid,0.550
